# Report Assets. Hero Sequence and Analysis Charts

Notebook ini menghasilkan aset visual untuk proposal, yaitu urutan hero raw frame ke prediksi akhir serta grafik analisis utama, mengikuti format aset situs konjungtiva agar konsisten lintas situs dalam dokumen proposal.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from configs import paths
from src.common import eval as evaluation, features, preprocess
from src.sites.palm import roi as palm_roi

output_dir = paths.outputs_dir("palm")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
frame_qc = pd.read_csv(output_dir / "frame_qc.csv")

## Pick Sample

In [ ]:
candidates = frame_qc[frame_qc["detected"]]["uid"]
sample_uid = candidates.sample(1, random_state=11).iloc[0]
sample_row = manifest[manifest["uid"] == sample_uid].iloc[0]
sample_frame_row = frame_qc[frame_qc["uid"] == sample_uid].iloc[0]
print(sample_uid)

## Stage 1. Raw Frame

Frame terbaik hasil pemilihan MediaPipe Hands dari video partisipan, sebelum segmentasi ROI.

In [ ]:
raw_frame = cv2.cvtColor(cv2.imread(sample_frame_row["frame_path"]), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(5, 5))
plt.imshow(raw_frame)
plt.title("Stage 1. Selected Raw Frame")
plt.axis("off")
plt.show()

## Stage 2. Landmark Detection

Dua puluh satu titik landmark MediaPipe Hands yang menjadi dasar convex hull region of interest.

In [ ]:
landmarks_px = np.load(sample_frame_row["landmarks_path"])
plt.figure(figsize=(5, 5))
plt.imshow(raw_frame)
plt.scatter(landmarks_px[:, 0], landmarks_px[:, 1], s=15, c="red")
plt.title("Stage 2. Hand Landmarks")
plt.axis("off")
plt.show()

## Stage 3. Palm ROI Segmentation

Region of interest hasil convex hull landmark yang diperhalus warna kulit YCbCr.

In [ ]:
result = palm_roi.palm_mask_from_landmarks(raw_frame, landmarks_px)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(result["mask"], cmap="gray")
axes[0].set_title("ROI Mask")
axes[1].imshow(result["rgb"])
axes[1].set_title("Cropped ROI")
for ax in axes:
    ax.axis("off")
plt.show()

## Stage 4. Illumination Normalization

CLAHE pada channel V menstabilkan kontras lokal ROI palm sebelum ekstraksi fitur.

In [ ]:
normalized = preprocess.normalize_roi(sample_row["roi_path"])
plt.figure(figsize=(5, 5))
plt.imshow(normalized["rgb"])
plt.title("Stage 4. Normalized ROI")
plt.axis("off")
plt.show()

## Stage 5. Color Biomarker Heatmap

Peta erythema index piksel demi piksel memvisualisasikan bagaimana model membaca sinyal pallor pada permukaan telapak tangan.

In [ ]:
rgb_float = normalized["rgb"].astype(np.float32)
mean_g = rgb_float[:, :, 1]
erythema_map = np.log10(1.0 / (mean_g / 255.0 + 1e-6))
erythema_map[~normalized["valid_mask"]] = np.nan

plt.figure(figsize=(5, 5))
plt.imshow(erythema_map, cmap="inferno")
plt.title("Stage 5. Erythema Index Heatmap")
plt.axis("off")
plt.colorbar(label="erythema index")
plt.show()

## Stage 6. Final Prediction Card

Ringkasan estimasi hemoglobin, klasifikasi anemia, dan severity untuk satu sampel, sebagai contoh keluaran akhir aplikasi.

In [ ]:
handcrafted_row = features.compute_handcrafted_features(normalized["rgb"], normalized["valid_mask"])
print("uid", sample_uid)
print("hemoglobin sebenarnya", sample_row["hb_gdl"], "g/dL")
print("severity sebenarnya", sample_row["severity"])
print("red_ratio", round(handcrafted_row["red_ratio"], 4))
print("erythema_index", round(handcrafted_row["erythema_index"], 4))

## Analysis Charts (Standalone Export)

Grafik perbandingan model dan Bland-Altman final untuk dilampirkan langsung ke proposal tanpa perlu menjalankan ulang notebook training.

In [ ]:
comparison_table = pd.read_csv(output_dir / "multitask_model_comparison.csv")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comparison_table["configuration"], comparison_table["mae"], color="teal")
ax.set_ylabel("MAE (g/dL)")
ax.set_title("Model Comparison, Hemoglobin MAE")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
oof_tuned = pd.read_csv(output_dir / "multitask_oof_full_fusion_tuned.csv")
bland_altman = evaluation.bland_altman_stats(oof_tuned["hb_true"], oof_tuned["hb_pred"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(bland_altman["mean_value"], bland_altman["difference"], alpha=0.4, s=10)
ax.axhline(bland_altman["bias"], color="black", label="bias")
ax.axhline(bland_altman["lower_limit"], color="red", linestyle="--", label="limits of agreement")
ax.axhline(bland_altman["upper_limit"], color="red", linestyle="--")
ax.set_title("Final Bland-Altman, Full Fusion Tuned")
ax.legend()
plt.tight_layout()
plt.show()